In [0]:
from pyspark.sql import functions as F

base_path = "/Volumes/healthcare_demo/bronze/raw"

patients = spark.read.option("header", True).option("inferSchema", True).csv(base_path + "/patients")
referrals = spark.read.option("header", True).option("inferSchema", True).csv(base_path + "/referrals")
appointments = spark.read.option("header", True).option("inferSchema", True).csv(base_path + "/appointments")
capacity = spark.read.option("header", True).option("inferSchema", True).csv(base_path + "/capacity")
referral_documents = spark.read.option("header", True).option("inferSchema", True).csv(base_path + "/referral_documents")

display(patients)
display(referrals)
display(appointments)
display(capacity)
display(referral_documents)

In [0]:
from delta.tables import DeltaTable

def merge_to_delta(source_df, target_table, merge_keys):

    field_names = source_df.schema.fieldNames()
    if "_ingested_at" not in field_names:
        source_df = source_df.withColumns({"_ingested_at": F.current_timestamp()})

    duplicate_keys = (
        source_df.groupBy(merge_keys)
        .count()
        .filter(F.col("count") > 1)
    )

    if duplicate_keys.limit(1).count() > 0:
        raise ValueError(f"Duplicate keys found in {target_table}")

    if not spark.catalog.tableExists(target_table):
        source_df.write.format("delta").saveAsTable(target_table)
        print(f"Created: {target_table}")
    else:
        delta_table = DeltaTable.forName(spark, target_table)

        condition = " AND ".join(
            [f"target.{key} = source.{key}" for key in merge_keys]
        )

        (
            delta_table.alias("target")
            .merge(source_df.alias("source"), condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        print(f"Merged: {target_table}")

In [0]:
merge_to_delta(patients, "healthcare_demo.bronze.patients", ["patient_id"])
merge_to_delta(referrals, "healthcare_demo.bronze.referrals", ["referral_id"])
merge_to_delta(appointments, "healthcare_demo.bronze.appointments", ["appointment_id"])
merge_to_delta(capacity, "healthcare_demo.bronze.capacity", ["location_id", "specialty"])
merge_to_delta(referral_documents, "healthcare_demo.bronze.referral_documents", ["document_id"])

In [0]:
documents = spark.table(
    "healthcare_demo.bronze.referral_documents"
)

print("Document count:", documents.count())
print("Columns:", documents.columns)

display(documents)

In [0]:
silver = spark.table(
    "healthcare_demo.silver.patient_referral_details"
)

document_check = (
    silver.select("referral_id").distinct()
    .join(
        documents.select("referral_id").distinct(),
        on="referral_id",
        how="left"
    )
)

display(document_check)

In [0]:
display(
    silver.groupBy(
        "clinical_symptoms",
        "document_priority",
        "document_specialty"
    ).count().orderBy("document_specialty")
)